# GAT RNA Motif Classifier

Graph Attention Network (GATv2) for GNRA tetraloop detection.

This notebook follows the **same pipeline** as `10-remove-redundancy-and-train-models.ipynb`:
- Input: `geometric_features.csv`
- Raw angle columns are dropped (sin/cos variants are kept)
- Splits are built from the same 6 clustering JSON files
  (approximate/exact × hierarchical/affinity-propagation/facility-location)
- One model is trained and saved per split

What differs from classical ML:
- Features are converted into graphs (distances → edges, sin/cos angles → node features)
- The classifier is a 3-layer GATv2 with global mean pooling

## Imports

In [1]:
import copy
import itertools
import json
import os
import random
from pickle import dump

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool

optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs("charts", exist_ok=True)


c:\Users\jmp\anaconda3\envs\GNN\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
SEED = 42
NUM_NODES = 8   # window size — matches geometric_features.csv (indices 0-7)
EDGE_DIM = 2    # distance value + is_consecutive flag
# NODE_DIM is auto-detected after the graph encoder is defined (see Cell 6)

# ── Optuna search space ─────────────────────────────────────────────────────────
#   These ranges are sampled per trial. The best completed trial's HP dict is
#   stored in HP after the study and reused for every split in the main loop.
#   If all trials are pruned, HP falls back to HP_DEFAULT (defined in the Optuna
#   cell) so the notebook always continues without crashing.
#
#   CPU tip: set OPTUNA_N_TRIALS=10 and OPTUNA_MAX_EPOCHS=40 for a quick run.
#   GPU tip: set OPTUNA_N_TRIALS=50 and OPTUNA_MAX_EPOCHS=100 for thorough search.
OPTUNA_N_TRIALS   = 15   # number of Optuna trials (raise for more thorough search)
OPTUNA_MAX_EPOCHS = 50   # epochs per trial (shorter than final training)
OPTUNA_PATIENCE   = 10   # early-stopping patience during HPO

# ── Final training settings (fixed, independent of HPO) ───────────────────────
FINAL_MAX_EPOCHS = 200
FINAL_PATIENCE   = 30     # early-stopping patience (epochs without MCC improvement)


def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


## Cluster Loading

Identical to `10-remove-redundancy-and-train-models.ipynb`.

In [3]:
class Cluster:
    def __init__(self, representative, members):
        self.representative = representative
        self.members = members

    def __repr__(self):
        return (
            f"Cluster(representative='{self.representative}', members={self.members})"
        )


def clean_name(name: str) -> str:
    return name.split("/")[-1].split(".")[0]


def load_clusters(array) -> list:
    result = []
    for obj in array:
        representative = clean_name(obj["representative"])
        members = list(map(clean_name, obj["members"]))
        result.append(Cluster(representative, members))
    return result


def load_all_clusters():
    for mode in ["approximate", "exact"]:
        for method in ["hierarchical", "affinity-propagation", "facility-location"]:
            path = f"{mode}-{method}.json"
            with open(path) as f:
                data = json.load(f)
                yield (mode, method, load_clusters(data["clustering"]["clusters"]))


clustering = {}
for mode, method, clusters in load_all_clusters():
    clustering[(mode, method)] = clusters

# Sanity-check cluster counts (same assertions as notebook 10)
assert len(clustering[("approximate", "hierarchical")]) == 88
assert len(clustering[("approximate", "affinity-propagation")]) == 34
assert len(clustering[("approximate", "facility-location")]) == 33
assert len(clustering[("exact", "hierarchical")]) == 82
assert len(clustering[("exact", "affinity-propagation")]) == 34
assert len(clustering[("exact", "facility-location")]) == 33

print(f"Loaded {len(clustering)} splits.")

Loaded 6 splits.


## Data Loading & Preprocessing

Same feature-dropping logic as notebook 10.

In [4]:
df = pd.read_csv("geometric_features.csv")

# Drop raw angle columns — keep sin/cos variants (identical to notebook 10)
columns_to_drop = []
for i, j, k in itertools.combinations(range(8), 3):
    columns_to_drop.append(f"a{i}{j}{k}")
for i, j, k, l in itertools.combinations(range(8), 4):
    columns_to_drop.append(f"t{i}{j}{k}{l}")

df_filtered = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
print(f"Original columns : {len(df.columns)}")
print(f"Filtered columns : {len(df_filtered.columns)}")

positive = df_filtered[ df_filtered["gnra"]]
negative = df_filtered[~df_filtered["gnra"]]
print(f"Positives: {len(positive)}  |  Negatives: {len(negative)}")

Original columns : 408
Filtered columns : 282
Positives: 315  |  Negatives: 1975


## Graph Encoding

Converts a flat feature row into a `torch_geometric.data.Data` object.

Mapping rules (derived from column-name digit parsing, 0-indexed):
- 2 digit indices → **edge** feature (distance `d{i}{j}`)
- 3 digit indices → **node** feature for the *middle* node (`sin_a{i}{j}{k}`, `cos_a{i}{j}{k}`)
- 4 digit indices → **node** feature for *both* middle nodes (`sin_t{i}{j}{k}{l}`, `cos_t{i}{j}{k}{l}`)

There is no nucleotide sequence in `geometric_features.csv`, so node features are
purely geometric (angle sin/cos). Terminal nodes (0 and 7) receive zero-padded
features because they never appear as angle/torsion midpoints.

Edges are made **undirected** by mirroring `(i→j)` and `(j→i)`.

In [5]:
def _digits(col: str) -> list[int]:
    """Extract all digit characters from a column name as a list of ints."""
    return [int(c) for c in col if c.isdigit()]


def row_to_graph(row: pd.Series, feat_cols: list[str]) -> Data:
    """
    Convert one row of the (scaled) feature DataFrame into a PyG Data object.

    Parameters
    ----------
    row       : a row from df_filtered (must already be scaled)
    feat_cols : list of feature column names (excludes 'source_file' and 'gnra')
    """
    edge_dict    = {}                            # (i, j) -> [distance, ...]
    node_angle   = {i: [] for i in range(NUM_NODES)}
    node_torsion = {i: [] for i in range(NUM_NODES)}

    for col in feat_cols:
        idxs = _digits(col)
        val  = float(row[col]) if not pd.isna(row[col]) else 0.0

        if len(idxs) == 2:
            i, j = idxs[0], idxs[1]
            # Guard against indices that exceed the window (shouldn't happen with 0-7)
            if i < NUM_NODES and j < NUM_NODES:
                edge_dict.setdefault((i, j), []).append(val)

        elif len(idxs) == 3:
            middle = idxs[1]          # the middle atom carries the angle feature
            if all(x < NUM_NODES for x in idxs):
                node_angle[middle].append(val)

        elif len(idxs) == 4:
            mid1, mid2 = idxs[1], idxs[2]
            if all(x < NUM_NODES for x in idxs):
                node_torsion[mid1].append(val)
                node_torsion[mid2].append(val)

    # ── Node feature matrix ───────────────────────────────────────────────────
    max_angle   = max((len(v) for v in node_angle.values()),   default=0)
    max_torsion = max((len(v) for v in node_torsion.values()), default=0)

    node_feats = []
    for i in range(NUM_NODES):
        a = node_angle[i]   + [0.0] * (max_angle   - len(node_angle[i]))
        t = node_torsion[i] + [0.0] * (max_torsion - len(node_torsion[i]))
        node_feats.append(a + t)

    x = torch.tensor(node_feats, dtype=torch.float32)   # shape [NUM_NODES, node_dim]

    # ── Edge index + attributes ───────────────────────────────────────────────
    edge_index_list: list = []
    edge_attr_list:  list = []

    for (i, j), weights in edge_dict.items():
        is_consecutive = 1.0 if abs(i - j) == 1 else 0.0
        # Keep only the first accumulated value (single distance per pair)
        # plus the is_consecutive flag → EDGE_DIM = 2
        edge_attr_list.append([weights[0], is_consecutive])
        edge_index_list.append([i, j])

    if edge_index_list:
        edge_attr  = torch.tensor(edge_attr_list,  dtype=torch.float32)
        edge_index = torch.tensor(edge_index_list, dtype=torch.int64).t().contiguous()
        # Make undirected
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_attr  = torch.cat([edge_attr,  edge_attr],          dim=0)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.int64)
        edge_attr  = torch.zeros((0, EDGE_DIM), dtype=torch.float32)

    y = torch.tensor([int(row["gnra"])], dtype=torch.int64)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

## Auto-detect NODE_DIM

Run a single sample through the encoder to discover the actual node feature dimension
rather than hardcoding it.

In [6]:
_feat_cols_all = [c for c in df_filtered.columns if c not in ("source_file", "gnra")]
_sample = row_to_graph(df_filtered.iloc[0], _feat_cols_all)

NODE_DIM = _sample.x.shape[1]
EDGE_DIM = _sample.edge_attr.shape[1] if _sample.edge_attr.numel() > 0 else 2

print(f"NODE_DIM = {NODE_DIM}")
print(f"EDGE_DIM = {EDGE_DIM}")

NODE_DIM = 84
EDGE_DIM = 2


## GATv2 Model Definition

In [7]:
class GATModel(torch.nn.Module):
    """
    3-layer GATv2 graph classifier.

    Architecture
    ------------
    conv1 : GATv2Conv(NODE_DIM  → hidden1,  heads=heads,  concat=True)
    conv2 : GATv2Conv(hidden1*h → hidden2,  heads=heads,  concat=True)
    conv3 : GATv2Conv(hidden2*h → hidden3,  heads=1,      concat=False)
    pool  : global mean pool  →  shape [batch, hidden3]
    lin   : Linear(hidden3, 2)

    hp keys
    -------
    hidden1, hidden2, hidden3 : layer output channels
    heads                     : attention heads (shared across conv1/conv2)
    dropout                   : dropout before the linear classifier
    """

    def __init__(self, hp: dict, node_dim: int, edge_dim: int):
        super().__init__()
        h1, h2, h3 = hp["hidden1"], hp["hidden2"], hp["hidden3"]
        heads = hp["heads"]

        self.conv1 = GATv2Conv(node_dim,    h1, edge_dim=edge_dim, heads=heads,  concat=True)
        self.conv2 = GATv2Conv(h1 * heads,  h2, edge_dim=edge_dim, heads=heads,  concat=True)
        self.conv3 = GATv2Conv(h2 * heads,  h3, edge_dim=edge_dim, heads=1,      concat=False)
        self.lin   = torch.nn.Linear(h3, 2)
        self.dropout_p = hp["dropout"]

    def forward(self, x, edge_index, edge_attr, batch):
        x = self.conv1(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr=edge_attr).relu()
        x = self.conv3(x, edge_index, edge_attr=edge_attr).relu()
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout_p, training=self.training)
        return self.lin(x)

## Training & Evaluation Helpers

In [8]:
def train_epoch(model, loader, optimizer, criterion) -> None:
    model.train()
    for data in loader:
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()


def evaluate(model, loader) -> tuple[float, np.ndarray, np.ndarray]:
    """Return (accuracy, predictions, labels) for the given DataLoader."""
    model.eval()
    all_preds, all_labels = [], []
    correct = 0

    with torch.no_grad():
        for data in loader:
            out  = model(data.x, data.edge_index, data.edge_attr, data.batch)
            pred = out.argmax(dim=1)
            correct += int((pred == data.y).sum())
            all_preds.extend(pred.cpu().numpy().tolist())
            all_labels.extend(data.y.cpu().numpy().tolist())

    accuracy = correct / len(loader.dataset)
    return accuracy, np.array(all_preds), np.array(all_labels)


def compute_metrics(labels: np.ndarray, preds: np.ndarray) -> dict:
    """Compute accuracy, precision, recall, F1 (macro), and MCC."""
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
        "mcc":       matthews_corrcoef(labels, preds),
    }


def save_training_chart(
    history: dict,
    split_label: str,
    out_dir: str = "charts",
) -> str:
    """
    Save a 5-panel PNG with per-epoch training curves.

    Parameters
    ----------
    history    : dict with keys 'accuracy', 'precision', 'recall', 'f1', 'mcc',
                 each mapping to a list of per-epoch float values.
    split_label: human-readable label used in the title and filename.
    out_dir    : directory where the PNG is written.

    Returns
    -------
    Path to the saved PNG.
    """
    metrics = ["accuracy", "precision", "recall", "f1", "mcc"]
    titles  = ["Accuracy", "Precision", "Recall", "F1 Score", "MCC"]
    colors  = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]

    epochs = list(range(1, len(history["accuracy"]) + 1))

    fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
    fig.suptitle(f"Training Curves — {split_label}", fontsize=13, fontweight="bold")

    for ax, metric, title, color in zip(axes, metrics, titles, colors):
        ax.plot(epochs, history[metric], color=color, linewidth=1.8)
        best_val = max(history[metric])
        best_ep  = history[metric].index(best_val) + 1
        ax.axvline(best_ep, color=color, linestyle="--", alpha=0.4,
                   label=f"Best: {best_val:.3f} (ep {best_ep})")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylim(-0.05, 1.05) if metric != "mcc" else ax.set_ylim(-1.05, 1.05)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    safe_label = split_label.replace(" / ", "-").replace(" ", "_")
    out_path   = os.path.join(out_dir, f"training_{safe_label}.png")
    fig.savefig(out_path, dpi=120)
    plt.close(fig)
    return out_path


## Optuna Hyperparameter Optimisation

Run a short study on the **first split** to find good hyperparameters.  
The best trial's `HP` dict is then reused for all six cluster-based splits
in the main training loop below.


In [9]:
# ── Optuna objective ────────────────────────────────────────────────────────────
def make_objective(train_loader, test_loader, node_dim, edge_dim, n_pos, n_neg):
    """
    Returns a closure that Optuna calls once per trial.
    Trains a short run and returns the best validation MCC.
    """
    def objective(trial: optuna.Trial) -> float:
        hp = {
            "hidden1":      trial.suggest_categorical("hidden1",      [16, 32, 64]),
            "hidden2":      trial.suggest_categorical("hidden2",      [32, 64, 128]),
            "hidden3":      trial.suggest_categorical("hidden3",      [16, 32, 64]),
            "heads":        trial.suggest_categorical("heads",        [1, 2, 4]),
            "dropout":      trial.suggest_float("dropout",            0.1, 0.5, step=0.05),
            "lr":           trial.suggest_float("lr",                 1e-4, 1e-2, log=True),
            "weight_decay": trial.suggest_float("weight_decay",       0.0,  1e-3, step=1e-4),
            "batch_size":   HP_BATCH,
            "max_epochs":   OPTUNA_MAX_EPOCHS,
            "patience":     OPTUNA_PATIENCE,
        }

        pos_weight    = n_neg / n_pos if n_pos > 0 else 1.0
        class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)
        criterion     = torch.nn.CrossEntropyLoss(weight=class_weights)

        set_all_seeds(SEED + trial.number)   # different seed per trial for diversity
        model     = GATModel(hp, node_dim, edge_dim)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"]
        )

        best_mcc   = -2.0
        no_improve = 0

        for epoch in range(1, OPTUNA_MAX_EPOCHS + 1):
            train_epoch(model, train_loader, optimizer, criterion)
            _, val_preds, val_labels = evaluate(model, test_loader)
            val_mcc = matthews_corrcoef(val_labels, val_preds)

            if val_mcc > best_mcc:
                best_mcc   = val_mcc
                no_improve = 0
            else:
                no_improve += 1

            # Only report to pruner — never raise TrialPruned based on pruner alone;
            # let early-stopping handle convergence so at least some trials complete.
            trial.report(val_mcc, epoch)

            if no_improve >= OPTUNA_PATIENCE:
                break   # early stop, but trial is considered COMPLETE with best_mcc

        return best_mcc

    return objective


# ── Default HP (used as fallback if every trial is pruned) ─────────────────────
HP_DEFAULT = {
    "hidden1":      32,
    "hidden2":      64,
    "hidden3":      32,
    "heads":         2,
    "dropout":      0.3,
    "lr":           1e-3,
    "weight_decay": 0.0,
}

# ── Build data for the first split (used only for HPO) ─────────────────────────
print("Building train/test data for the first split (used for Optuna HPO) …")

_first_key    = next(iter(clustering))
_first_clust  = clustering[_first_key]
_mode, _meth  = _first_key

_clusters_sorted = sorted(_first_clust, key=lambda c: len(c.members))
_pos_test_names: list = []
for _cl in _clusters_sorted:
    _pos_test_names.extend([_cl.representative] + _cl.members)
    if len(_pos_test_names) >= 0.25 * len(positive):
        break

_positive_train = positive[~positive["source_file"].isin(_pos_test_names)]
_positive_test  = positive[ positive["source_file"].isin(_pos_test_names)]
_negative_train, _negative_test = train_test_split(negative, test_size=0.25, random_state=42)

_df_train = pd.concat([_positive_train, _negative_train]).reset_index(drop=True)
_df_test  = pd.concat([_positive_test,  _negative_test ]).reset_index(drop=True)

_feat_cols = [c for c in _df_train.columns if c not in ("source_file", "gnra")]
_scaler    = StandardScaler()
_df_train  = _df_train.copy(); _df_test = _df_test.copy()
_df_train[_feat_cols] = _scaler.fit_transform(_df_train[_feat_cols])
_df_test[_feat_cols]  = _scaler.transform(_df_test[_feat_cols])

_train_ds = [row_to_graph(row, _feat_cols) for _, row in _df_train.iterrows()]
_test_ds  = [row_to_graph(row, _feat_cols) for _, row in _df_test.iterrows()]

HP_BATCH      = 32
_train_loader = DataLoader(_train_ds, batch_size=HP_BATCH, shuffle=True)
_test_loader  = DataLoader(_test_ds,  batch_size=HP_BATCH)

_n_pos = int(_df_train["gnra"].sum())
_n_neg = int((~_df_train["gnra"]).sum())

# ── Run Optuna study ────────────────────────────────────────────────────────────
print(f"Starting Optuna study ({OPTUNA_N_TRIALS} trials, no aggressive pruning) …")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    # PatientPruner wraps NopPruner: only prune if a trial stalls for
    # OPTUNA_PATIENCE steps with no improvement — same budget as early stopping,
    # so effectively disables premature pruning while still allowing Optuna infra.
    pruner=optuna.pruners.PatientPruner(optuna.pruners.NopPruner(), patience=OPTUNA_PATIENCE),
)
study.optimize(
    make_objective(_train_loader, _test_loader, NODE_DIM, EDGE_DIM, _n_pos, _n_neg),
    n_trials=OPTUNA_N_TRIALS,
    show_progress_bar=True,
)

# ── Pick best completed trial (fall back to defaults if all were pruned) ────────
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

if completed:
    best_trial = max(completed, key=lambda t: t.value)
    print(f"\nBest trial #{best_trial.number}  |  MCC = {best_trial.value:.4f}")
    print("  Params:")
    for k, v in best_trial.params.items():
        print(f"    {k:>15s} = {v}")
    best_params = best_trial.params
else:
    print("\n⚠  No trials completed (all pruned). Falling back to default HP.")
    best_params = HP_DEFAULT

# ── Assemble final HP dict ──────────────────────────────────────────────────────
HP = {
    **best_params,
    "batch_size":  HP_BATCH,
    "max_epochs":  FINAL_MAX_EPOCHS,
    "patience":    FINAL_PATIENCE,
}
print(f"\nHP dict for final training: {HP}")


Building train/test data for the first split (used for Optuna HPO) …


C:\Users\jmp\AppData\Local\Temp\ipykernel_27164\3288563293.py:114: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  pruner=optuna.pruners.PatientPruner(optuna.pruners.NopPruner(), patience=OPTUNA_PATIENCE),


Starting Optuna study (15 trials, no aggressive pruning) …


Best trial: 2. Best value: 0.992804: 100%|██████████| 15/15 [06:47<00:00, 27.13s/it]


Best trial #2  |  MCC = 0.9928
  Params:
            hidden1 = 16
            hidden2 = 64
            hidden3 = 64
              heads = 4
            dropout = 0.1
                 lr = 0.006586289317583112
       weight_decay = 0.0002

HP dict for final training: {'hidden1': 16, 'hidden2': 64, 'hidden3': 64, 'heads': 4, 'dropout': 0.1, 'lr': 0.006586289317583112, 'weight_decay': 0.0002, 'batch_size': 32, 'max_epochs': 200, 'patience': 30}


## Main Training Loop

Iterates over all 6 cluster-based splits. Uses the **Optuna-tuned HP** from the study above. Per-epoch metrics (accuracy, precision, recall, F1, MCC) are recorded and saved as PNG charts in `charts/`.


In [10]:
set_all_seeds(SEED)

summary_rows = []   # collects final metrics per split for a summary table

for (mode, method), clusters in clustering.items():
    print(f"\n{'='*60}")
    print(f"  Split: {mode} / {method}")
    print(f"{'='*60}")

    # ── Build train/test split ─────────────────────────────────────────────────
    clusters_sorted = sorted(clusters, key=lambda c: len(c.members))

    positive_test_names: list = []
    for cluster in clusters_sorted:
        positive_test_names.extend([cluster.representative] + cluster.members)
        if len(positive_test_names) >= 0.25 * len(positive):
            break

    positive_train = positive[~positive["source_file"].isin(positive_test_names)]
    positive_test  = positive[ positive["source_file"].isin(positive_test_names)]
    negative_train, negative_test = train_test_split(
        negative, test_size=0.25, random_state=42
    )

    df_train = pd.concat([positive_train, negative_train]).reset_index(drop=True)
    df_test  = pd.concat([positive_test,  negative_test ]).reset_index(drop=True)

    print(f"  Train: {len(df_train)} samples  "
          f"({df_train['gnra'].sum()} pos / {(~df_train['gnra']).sum()} neg)")
    print(f"  Test : {len(df_test)}  samples  "
          f"({df_test['gnra'].sum()} pos / {(~df_test['gnra']).sum()} neg)")

    # ── Scale features ─────────────────────────────────────────────────────────
    feat_cols = [c for c in df_train.columns if c not in ("source_file", "gnra")]

    scaler = StandardScaler()
    df_train = df_train.copy()
    df_test  = df_test.copy()
    df_train[feat_cols] = scaler.fit_transform(df_train[feat_cols])
    df_test[feat_cols]  = scaler.transform(df_test[feat_cols])

    # ── Convert rows to graphs ─────────────────────────────────────────────────
    train_dataset = [row_to_graph(row, feat_cols) for _, row in df_train.iterrows()]
    test_dataset  = [row_to_graph(row, feat_cols) for _, row in df_test.iterrows()]

    train_loader = DataLoader(train_dataset, batch_size=HP["batch_size"], shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=HP["batch_size"])

    # ── Class-weighted loss ────────────────────────────────────────────────────
    n_pos = int(df_train["gnra"].sum())
    n_neg = int((~df_train["gnra"]).sum())
    pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
    print(f"  pos_weight: {pos_weight:.2f}x  (neg {n_neg} / pos {n_pos})")

    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

    # ── Model + optimiser ──────────────────────────────────────────────────────
    set_all_seeds(SEED)
    model     = GATModel(HP, NODE_DIM, EDGE_DIM)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"]
    )

    best_mcc   = -2.0
    best_state = None
    no_improve = 0

    # History buffers — filled each epoch for the training chart
    history = {m: [] for m in ("accuracy", "precision", "recall", "f1", "mcc")}

    # ── Training loop with early stopping on MCC ───────────────────────────────
    for epoch in range(1, HP["max_epochs"] + 1):
        train_epoch(model, train_loader, optimizer, criterion)
        _, val_preds, val_labels = evaluate(model, test_loader)

        m = compute_metrics(val_labels, val_preds)
        for key, val in m.items():
            history[key].append(val)

        val_mcc = m["mcc"]
        if val_mcc > best_mcc:
            best_mcc   = val_mcc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 20 == 0:
            print(f"  Epoch {epoch:03d} | MCC {val_mcc:.4f} (best {best_mcc:.4f})")

        if no_improve >= HP["patience"]:
            print(f"  Early stopping at epoch {epoch} (best MCC {best_mcc:.4f})")
            break

    # ── Save training chart ────────────────────────────────────────────────────
    split_label = f"{mode} / {method}"
    chart_path  = save_training_chart(history, split_label)
    print(f"  Chart saved → {chart_path}")

    # ── Final evaluation ───────────────────────────────────────────────────────
    model.load_state_dict(best_state)
    _, final_preds, final_labels = evaluate(model, test_loader)
    final_metrics = compute_metrics(final_labels, final_preds)

    print(f"\n  Classifier: GAT  (split: {split_label})")
    print(classification_report(final_labels, final_preds))

    summary_rows.append({
        "split": split_label,
        **final_metrics,
    })

    # ── Save model ─────────────────────────────────────────────────────────────
    model_name = "gat"
    payload = {
        "classifier_name": "GAT",
        "model_state_dict": best_state,
        "hp":              HP,
        "scaler":          scaler,
        "feature_columns": feat_cols,
        "node_dim":        NODE_DIM,
        "edge_dim":        EDGE_DIM,
        "num_nodes":       NUM_NODES,
        "window_size":     NUM_NODES,
        "positive_label":  True,
        "split_mode":      mode,
        "split_method":    method,
    }
    out_path = f"{mode}-{method}-{model_name}.pkl"
    with open(out_path, "wb") as f:
        dump(payload, f)
    print(f"  Saved → {out_path}")

# ── Print cross-split summary ──────────────────────────────────────────────────
print("\n" + "="*60)
print("  CROSS-SPLIT SUMMARY")
print("="*60)
summary_df = pd.DataFrame(summary_rows).set_index("split")
summary_df = summary_df[["accuracy", "precision", "recall", "f1", "mcc"]]
print(summary_df.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nMean across splits:")
print(summary_df.mean().to_string(float_format=lambda x: f"{x:.4f}"))



  Split: approximate / hierarchical
  Train: 1715 samples  (234 pos / 1481 neg)
  Test : 575  samples  (81 pos / 494 neg)
  pos_weight: 6.33x  (neg 1481 / pos 234)
  Epoch 020 | MCC 0.9648 (best 0.9856)
  Epoch 040 | MCC 0.9350 (best 0.9856)
  Epoch 060 | MCC 0.9117 (best 0.9856)
  Early stopping at epoch 60 (best MCC 0.9856)
  Chart saved → charts\training_approximate-hierarchical.png

  Classifier: GAT  (split: approximate / hierarchical)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       494
           1       0.99      0.99      0.99        81

    accuracy                           1.00       575
   macro avg       0.99      0.99      0.99       575
weighted avg       1.00      1.00      1.00       575

  Saved → approximate-hierarchical-gat.pkl

  Split: approximate / affinity-propagation
  Train: 1717 samples  (236 pos / 1481 neg)
  Test : 573  samples  (79 pos / 494 neg)
  pos_weight: 6.28x  (neg 1481 / pos 236)
  Epoch 020

## Summary Chart

Grouped bar chart comparing all five metrics across the six cluster-based splits.


In [11]:
fig, ax = plt.subplots(figsize=(14, 5))

metrics_to_plot = ["accuracy", "precision", "recall", "f1", "mcc"]
colors          = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
x               = np.arange(len(summary_df))
width           = 0.15

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    offset = (i - 2) * width
    bars   = ax.bar(x + offset, summary_df[metric], width, label=metric.upper(),
                    color=color, alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=7, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(summary_df.index, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_ylim(-0.1, 1.15)
ax.set_title("GAT — Final Metrics per Split", fontweight="bold", fontsize=13)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
ax.axhline(0, color="black", linewidth=0.8)

fig.tight_layout()
summary_chart_path = "charts/summary_all_splits.png"
fig.savefig(summary_chart_path, dpi=120)
plt.close(fig)
print(f"Summary chart saved → {summary_chart_path}")


Summary chart saved → charts/summary_all_splits.png
